In [1]:
# Imports
from dotenv import load_dotenv
from anthropic import Anthropic
from building_with_the_claude_api import add_user_message, add_assistant_message, chat, Effort
from building_with_the_claude_api.prompt_evaluator import PromptEvaluator, generate_prompt_evaluation_report


In [2]:
# Client Initialization and helper functions

load_dotenv()

client = Anthropic()

# model = "claude-haiku-4-5"
model = "claude-sonnet-4-6"


In [3]:
# Create an instance of PromptEvaluator
# Increase `max_concurrent_tasks` for greater concurrency, but beware of rate limit errors!
evaluator = PromptEvaluator(max_concurrent_tasks=1)

In [4]:
dataset = evaluator.generate_dataset(
    client=client,
    model=model,
    # Describe the purpose or goal of the prompt you're trying to test
    task_description="write a compact, concise 1 day meal plan for a single athlete",
    # Describe the different inputs that your prompt requires
    prompt_inputs_spec={
        "height": "Athlete height in cm",
        "weight": "Athlete weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete",
    },
    # Number of test cases to generate (recommend keeping this low if you're getting rate limit errors)
    num_cases=3,
)

Generated 1/3 test cases
Generated 2/3 test cases
Generated 3/3 test cases


In [5]:
# Define and run the prompt you want to evaluate, returning the raw model output
# This function is executed once for each test case
def run_prompt(prompt_inputs):
    prompt = f"""
    What should this person eat?

    - Height: {prompt_inputs["height"]}
    - weight: {prompt_inputs["weight"]}
    - Goal: {prompt_inputs["goal"]}
    - Dietary restrictions:  {prompt_inputs["restrictions"]}
    """

    messages = []
    add_user_message(messages, prompt)
    return chat(messages=messages, client=client, model=model, effort=Effort.LOW)


In [6]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt, client=client, model=model, extra_criteria = """
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions and timing
    """
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 2.3333333333333335
